# Research question. What structure exists in Fashion-MNIST, how can different machine-learning models exploit that structure, and what do their successes and failures teach us about the data and its representations?

## Part 0 - Setup: data and computing

### 0.1 Fixed Data Partitions

1. First load the data, 60,000 official train and 10,000 official test data. 
2. Then split the official training set, stratified and seeded: 50,000 train and 10,000 validation. 
    - Stratification means splitting or sampling data so that each resulting partition or subset preserves approx the same class proportions as the source data. Randomly select examples within each class
    - For example, if each of the ten classes represents 10% of the source data, a stratified 10,000 image subset contains approx 1,000 images per class. Use a fixed random seed and save the selected indices so that your splits are reproducible.
3. Save the seed and indices
4. CPU Friendly option, fit models on a fixed stratified 10,000 image subset of the training partition; keep the same validation and test sets. 

In [1]:
# Installations
%pip install scikit-learn
%pip install numpy
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Imports
import numpy as np
import sklearn
from sklearn import model_selection

#For FashionMNIST dataset
import torchvision
import torch
from torchvision.datasets import FashionMNIST

#Fixed seed
seed = 0

**1. Load the training data.**

In [3]:
#Make sure to have windows defender protection off
training_data = FashionMNIST(root="data", train=True, download=True)
testing_data = FashionMNIST(root="data", train=False, download=True)

#Get the data and labels from the dataset
x = training_data.data.numpy()
y = training_data.targets.numpy()

#Same for testing data (untouched until final eval)
x_test = testing_data.data.numpy()
y_test = testing_data.targets.numpy()



**2. Split the offficial training set. | 3. CPU-friendly option**
Must also save the indices of each 

In [4]:
#Create list of indices for subsets
indices = np.arange(len(y))

#50000 train and 10000 validation. Fixed seed for reproducibility. stratify = y ensures that the class distribution is proportional to the original dataset.
x_train, x_val, y_train, y_val, train_indices, val_indices = model_selection.train_test_split(x, y, indices, test_size=1/6, random_state=seed, stratify=y)

#CPU-friend option: stratified 10,000-image subset of the training parition for faster training.
x_train2, _, y_train2, _, train_indices2, _ = model_selection.train_test_split(x_train, y_train, train_indices, train_size = 0.2, random_state = seed, stratify=y_train)


**4. Cluster / Feature-extraction subset.**

In [5]:
#Create test indices for 1000 test images. 
test_indices = np.arange(len(y_test))

#Create fixed 3000 train, 1000 validation and 1000 test images
x_cluster_train, _, y_cluster_train, _, cluster_train_indices, _ = model_selection.train_test_split(x_train2, y_train2, train_indices2, train_size = 3000, random_state = seed, stratify=y_train2)
x_cluster_val, _, y_cluster_val, _, cluster_val_indices, _ = model_selection.train_test_split(x_val, y_val, val_indices, train_size = 1000, random_state = seed, stratify=y_val)
x_cluster_test, _, y_cluster_test, _, cluster_test_indices, _ = model_selection.train_test_split(x_test, y_test, test_indices, train_size = 1000, random_state = seed, stratify=y_test)

**5. Save the seed and indices.** (note: the numbers refer to the instruction order according to the brief, not ordered according to what is run on this file hence 3 is lower than 5)

In [6]:
#Save as numpy file 
np.savez(
    'splits/indices.npz',
    train_indices=train_indices, #50 000 training indices
    val_indices=val_indices, #10 000 validation indices
    subset_indices=train_indices2, #10 000 training subset indices
    seed=seed
)

#Save clustering indices
np.savez(
    'splits/clustering_indices.npz',
    train_indices=cluster_train_indices,
    val_indices=cluster_val_indices,
    test_indices=cluster_test_indices,
    seed=seed
)

In [16]:
#Check cluster and subset shapes
print(x_cluster_train.shape)
print(x_cluster_val.shape)
print(x_cluster_test.shape)
print(x_train.shape)
print(x_train2.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)

(3000, 28, 28)
(1000, 28, 28)
(1000, 28, 28)
(50000, 28, 28)
(10000, 28, 28)
(10000, 28, 28)
(10000, 28, 28)
(50000,)


### 0.2 Where to run your code | My Computer Environment

On a jupyter notebook (ipynb file) on VScode on my personal computer

- Operating system: Windows 10 Pro
- CPU: AMD Ryzen 7 5800X — 8 cores, 16 threads
- RAM: 32 GB
- GPU: NVIDIA GeForce RTX 4060 Ti — 8 GB VRAM

I am using the CPU-friendly option (10,000 image subset of the training partition) so GPU isn't necessary however, I still included it just in case.

I will record approximate training, extraction and inference times. 

Then cache extracted features instead of recomputing them.

### Cell for Determining Requirements

In [11]:

print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

numpy: 2.5.4
scikit-learn: 1.9.1
torch: 2.15.0+cpu
torchvision: 0.29.2+cpu


# Part 1 - Classical Classification [10 marks]

Guiding question: How do different classifiers use the same data, and what can their errors tell us?
## What you must do
**1.2 Metrics.** Notebook overall accuracy and a class-level view of performance, with enough information for readers to interpret any proportions.

**1.3 Error analysis.** Inspect a small set of correct and incorrect predictions. Discuss easy and hard classes.

**1.4 Complexity study.** Vary parameters that control model complexity and explain how model complexity affects generalization.

**1.5 Decisions and costs.** Define a binary decision problem of your own, derived from the ten classes. Using validation data, show how the decision threshold trades one error against the other.


### **1.1 Models.** Choose at least two classifiers or configurations from the methods taught in the module. Justify your choice for this data.


Two classifiers: multinomial logistic regression and decision trees.
Since we are working with 10 classes, both classifiers need to be able to classify k > 2 classes (in which cases binary logistic regression wouldn't work).
Logistic regression and decision trees are both multiclass classifiers, so therefore they are viable for this data.
Further, logistic regression constructs linear boundaries and produces probabilities for each class whereas decision trees can construct non-linear boundaries, and classifies based on a sequence of questions which are two different methods of classifying. 
They also have two different approaches to dealing with complexity and overfitting, for example regularisation for logistic regression and maximum depth for the decision tree. 
Comparing these classifiers on the same data allows me to investigate how their different prediction methods and complexity controls affect training and validation performance.


In [ ]:
# First flatten the images to 1D arrays (10,000, 28, 28) -> (10,000, 784)
x_flat_train = x_train2.reshape(x_train2.shape[0], -1)
x_flat_val = x_val.reshape(x_val.shape[0], -1)
x_flat_test = x_test.reshape(x_test.shape[0], -1)